# Stage 06 — Master Data: Part Master, Supersession & Catalogues
**Dashboard pages:** Part Master · Catalogues
**Outputs:** part_master.parquet · supersession_map.parquet · catalog_parts.parquet

**Supersession rule:** Build directed graph requested_PN→latest_PN; walk to terminal node (depth ≤20); guard cycles.
**PDF extraction:** pdfplumber (primary) → pymupdf+pytesseract (fallback for scanned pages).

In [ ]:
import sys, warnings, re
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

INTERIM   = PROJECT_ROOT / "data" / "interim"
PROCESSED = PROJECT_ROOT / "data" / "processed"
OUTPUTS   = PROJECT_ROOT / "data" / "outputs"

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams.update({
    "figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})
PALETTE = ["#4361EE","#EF4444","#2CC56F","#F59E0B","#A855F7",
           "#64748B","#06B6D4","#F97316","#10B981","#8B5CF6"]
STATUS_COLORS = {"stockout":"#EF4444","critical":"#F97316",
                 "low":"#F59E0B","ok":"#2CC56F","excess":"#4361EE"}
TIER_COLORS   = {"critical":"#EF4444","managed":"#F97316",
                 "watch":"#F59E0B","rationalise":"#94A3B8"}

def load(name, base=None):
    if base:
        p = Path(base) / name
        if p.exists(): return pd.read_parquet(p)
    for b in [INTERIM, PROCESSED, OUTPUTS]:
        p = b / name
        if p.exists(): return pd.read_parquet(p)
    raise FileNotFoundError(f"{name} not found")

def fmt_lkr(v):
    if abs(v) >= 1e9: return f"LKR {v/1e9:.1f}B"
    if abs(v) >= 1e6: return f"LKR {v/1e6:.1f}M"
    return f"LKR {v:,.0f}"


In [ ]:
pm  = load("part_master.parquet")
ss  = load("supersession_map.parquet") if (INTERIM/"supersession_map.parquet").exists() else pd.DataFrame()
cat = load("catalog_parts.parquet")    if (INTERIM/"catalog_parts.parquet").exists() else pd.DataFrame()
cpm = load("catalogue_part_master.parquet") if (INTERIM/"catalogue_part_master.parquet").exists() else pd.DataFrame()

print(f"Part master         : {len(pm):,} SKUs | cols: {pm.columns.tolist()}")
print(f"Supersession map    : {len(ss):,} rows")
print(f"Catalog parts       : {len(cat):,} rows")
print(f"Catalogue part master: {len(cpm):,} rows")


## Part Master Table

In [ ]:
print(pm.head(10).to_string())
print()
# Coverage stats
if "stock" in pm.columns: print(f"With stock > 0 : {(pm['stock']>0).sum():,}")
if "order_qty" in pm.columns: print(f"With order qty > 0 : {(pm['order_qty']>0).sum():,}")
if "compatible_models" in pm.columns:
    pm_models = pm["compatible_models"].str.split(",",expand=False).explode().str.strip()
    print(f"
Compatible models in part master:")
    print(pm_models.value_counts().head(10).to_string())

fig,axes = plt.subplots(1,2,figsize=(14,4))
if "order_qty" in pm.columns and "stock" in pm.columns:
    valid = pm[(pm["order_qty"]>0)|(pm["stock"]>0)]
    axes[0].scatter(pm["stock"].clip(upper=pm["stock"].quantile(0.95)),
                    pm["order_qty"].clip(upper=pm["order_qty"].quantile(0.95)),
                    alpha=0.3,s=12,color=PALETTE[0])
    axes[0].set_title("Part Master: Stock vs Order Qty"); axes[0].set_xlabel("Stock"); axes[0].set_ylabel("Order qty")
if "eod_rate" in pm.columns:
    pm["eod_rate"].hist(bins=50,ax=axes[1],color=PALETTE[3],edgecolor="white",alpha=0.8)
    axes[1].set_title("EOD Rate Distribution (End-of-Day sell-through rate)")
plt.tight_layout(); plt.show()


## Supersession Chain Analysis

In [ ]:
if len(ss)>0:
    print("Supersession columns:", ss.columns.tolist())
    print(ss.head(10).to_string())
    hops_col = next((c for c in ["hops","chain_length","depth"] if c in ss.columns),None)
    if hops_col:
        hops = ss[hops_col].value_counts().sort_index()
        fig,ax = plt.subplots(figsize=(9,4))
        hops.plot(kind="bar",ax=ax,color=PALETTE[3],edgecolor="white")
        ax.set_title("Supersession Chain Length (hops)
1=direct A->B; 2=A->B->C; etc.")
        ax.set_xlabel("Hops"); ax.set_ylabel("Count")
        plt.tight_layout(); plt.show()
        print(f"
Max chain length : {ss[hops_col].max()} hops")
        print(f"Direct (1-hop)   : {(ss[hops_col]==1).sum():,} ({(ss[hops_col]==1).mean()*100:.1f}%)")
else:
    print("Supersession map not found — run Stage 6 from the Pipeline page.")


## Catalogue Coverage

In [ ]:
from src.features.part_type_classifier import classify_series, CATEGORIES

use_df = cpm if len(cpm)>0 else (cat if len(cat)>0 else pm)
desc_col = "description" if "description" in use_df.columns else None

if desc_col:
    use_df = use_df.copy()
    use_df["part_type"] = classify_series(use_df[desc_col])
    counts = use_df["part_type"].value_counts().reindex(CATEGORIES,fill_value=0)
    COLOR_MAP = {"Engine":"#EF4444","Electrical":"#F59E0B","Wear":"#F97316","Service":"#2CC56F",
                 "Crash":"#4361EE","Cosmetic":"#A855F7","Fasteners":"#64748B","Unclassified":"#94A3B8"}
    bar_colors = [COLOR_MAP.get(c,"#94A3B8") for c in counts.index]

    fig,axes = plt.subplots(1,2,figsize=(14,4))
    counts.plot(kind="bar",ax=axes[0],color=bar_colors,edgecolor="white")
    axes[0].set_title("Part Type Classifier Distribution
"
                      "(Priority: Engine>Electrical>Wear>Service>Crash>Cosmetic>Fasteners)")
    axes[0].set_ylabel("SKUs"); axes[0].tick_params(axis="x",rotation=0)
    for bar,val in zip(axes[0].patches,counts.values):
        axes[0].text(bar.get_x()+bar.get_width()/2,bar.get_height()+3,str(val),ha="center",fontsize=8)
    counts.plot(kind="pie",ax=axes[1],colors=bar_colors,autopct="%1.0f%%",startangle=90,legend=False)
    axes[1].set_title("Part Type Share"); axes[1].set_ylabel("")
    plt.tight_layout(); plt.show()
    classified_pct = (1-counts["Unclassified"]/len(use_df))*100
    print(f"Classification coverage : {classified_pct:.1f}%")
    print(counts.to_string())

# Model coverage
model_col = next((c for c in ["compatible_models","model","Model"] if c in use_df.columns),None)
if model_col:
    mc = use_df[model_col].value_counts().head(15)
    fig,ax = plt.subplots(figsize=(11,4))
    mc.plot(kind="bar",ax=ax,color=PALETTE[0],edgecolor="white")
    ax.set_title("Parts per Model — Catalogue Coverage"); ax.set_ylabel("Parts")
    ax.tick_params(axis="x",rotation=45); plt.tight_layout(); plt.show()
